In [109]:
import numpy as np
import time
import pandas as pd
from IPython.display import display

# Zad 1 - Metoda Gaussa Jordana

$
Ax = b \\
$

$
[A | b]
$

Eliminacja gaussa

$
cond(A) = ||A|| \times ||A^{-1}||
$

||A|| - maksymalna wartość w macierzy

- _partial pivoting_ - szukamy maks elementu w kolumnie, którą zerujemy 
- _full pivoting_ - szukamy w podmacierzy (powoduje zamianę kolejności zmiennych)
- _scaling_ - dzielimy kolumny przez największą wartość bezwzględną

In [110]:
n = 4

A = np.random.rand(n, n).astype(np.float64)
b = np.random.rand(n).astype(np.float64)

In [111]:
def gauss_jordan(A: np.array, b: np.array) -> np.float64:
    """
    Solves given Ax = B system of linear equations with Gauss-Jordan 
    method and complete pivoting.
    """

    n, _ = A.shape
    A = np.hstack((A, b.reshape((n, 1)))).astype(np.float64)
    vars = [i for i in range(n)]

    for i in range(n):
        # Complete pivoting

        submatrix = A[i:, i:-1]

        sub_row, sub_col = np.unravel_index(np.argmax(np.abs(submatrix), axis=None), submatrix.shape)
        pivot_row = sub_row + i
        pivot_col = sub_col + i

        A[[i, pivot_row]] = A[[pivot_row, i]]

        if pivot_col != i:
            vars[pivot_col], vars[i] = vars[i], vars[pivot_col]
            A[:, [i, pivot_col]] = A[:, [pivot_col, i]]

        # Gauss

        factor = A[i, i]
        A[i] /= factor

        for row in range(n):
            if row == i:
                continue

            A[row] -= A[i] * A[row, i]
            A[row, i] = 0

        # Fix order of variables

        result = np.empty_like(A[:, -1])
        result[vars] = A[:, -1]

    return result


In [112]:
gauss_jordan(A, b), np.linalg.solve(A, b)

(array([-0.30506183,  1.6000726 , -1.03625227,  0.65734897]),
 array([-0.30506183,  1.6000726 , -1.03625227,  0.65734897]))

In [113]:
results = []

for n in range(500, 1001, 100):
    print(n)
    A = np.random.rand(n, n).astype(np.float32)
    b = np.random.rand(n).astype(np.float32)

    our_start = time.perf_counter()
    got = gauss_jordan(A, b)
    our_end = time.perf_counter()

    want_start = time.perf_counter()
    want = np.linalg.solve(A, b) 
    want_end = time.perf_counter()

    diff = want - got
    
    our_time = our_end - our_start
    want_time = want_end - want_start
    max_abs_error = np.max(np.abs(diff))

    results.append({
        "n": n,
        "our time (s)": our_time,
        "numpy time (s)": want_time,
        "max abs error": max_abs_error,
        "got": np.round(got[:5], 5),
        "want": np.round(want[:5], 5),
        "allclose": np.allclose(got, want)
    })

df = pd.DataFrame(results)

styled_df = df.style.format({
    "Custom Time (s)": "{:.4f}",
    "NumPy Time (s)": "{:.6f}",
    "Max Abs Error": "{:.2e}" 
}).set_table_styles([{
      'selector': 'caption',
      'props': [('font-size', '16px'), ('font-weight', 'bold')]
  }])

display(styled_df)

500
600
700
800
900
1000


,n,our time (s),numpy time (s),max abs error,got,want,allclose
0,500,0.316895,0.001760,0.000000,[-0.82326 -0.13376 0.06692 -2.05357 -0.04181],[-0.82326 -0.13376 0.06692 -2.05357 -0.04181],True
1,600,0.470963,0.002413,0.000002,[ 6.92581 1.52543 3.77835 -9.81135 -10.02558],[ 6.92581 1.52543 3.77835 -9.81134 -10.02558],True
2,700,0.660832,0.003267,0.000000,[-0.90091 1.49264 -0.22693 2.15438 -1.89827],[-0.90091 1.49264 -0.22693 2.15438 -1.89827],True
3,800,0.912473,0.003895,0.000000,[-0.24411 0.37398 0.67971 -0.7069 -1.04945],[-0.24411 0.37398 0.67971 -0.7069 -1.04945],True
4,900,1.300789,0.006829,0.000000,[ 0.07303 -1.09545 -0.77106 1.63592 3.42793],[ 0.07303 -1.09545 -0.77106 1.63592 3.42793],True
5,1000,1.576271,0.006316,0.000000,[ 0.84166 0.26827 -0.34467 -0.01309 0.75929],[ 0.84166 0.26827 -0.34467 -0.01309 0.75929],True
